[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR-GITHUB-USERNAME/JAXCode/blob/master/templates/06_mha.ipynb)

# 🔴 Hard: Multi-Head Attention

*Attention & Transformers*
Implement **multi-head attention**.

$$\text{head}_i = \text{softmax}\!\left(\frac{Q W^Q_i (K W^K_i)^\top}{\sqrt{d_k}}\right)V W^V_i$$

$$\text{MHA}(Q,K,V) = \text{Concat}(\text{head}_1..\text{head}_H)\,W^O$$

### Signature
```python
class MultiHeadAttention(nnx.Module):
    def __init__(self, d_model: int, num_heads: int, *, rngs: nnx.Rngs): ...
    def __call__(self, Q, K, V): ...
```

### Requirements
- Use `nnx.Linear(d_model, d_model)` for `self.W_q`, `self.W_k`, `self.W_v`, `self.W_o`
- `self.d_k = d_model // num_heads`
- `Q` is `(B, seq_q, d_model)`; `K` and `V` are `(B, seq_k, d_model)`
- Must support **cross-attention** — `seq_q != seq_k`
- Do **not** use `nnx.MultiHeadAttention`

`nnx.Linear` is an allowed building block: you are implementing attention, not
the projection. It also brings its own initialization.

### Heads are a reshape, not a loop
The whole trick is that $H$ separate attention computations are one batched
computation. `(B, S, d_model)` reshapes to `(B, S, H, d_k)` and transposes to
`(B, H, S, d_k)`, after which the head axis is just another batch axis and the
same einsum handles all of them. Nothing is looped, and the parameter count is
identical to single-head attention with the same `d_model` — you are
partitioning the projection, not adding to it.

### Why Q, K and V are separate arguments
Passing one `x` would only ever give you self-attention. Taking three inputs
means the identical class does self-attention (`mha(x, x, x)`) and
cross-attention (`mha(decoder, encoder, encoder)`), which is exactly how an
encoder-decoder transformer reuses one implementation.

### The trap
It is tempting to read a single `S` off `Q` and use it for `K` too. That works
for every self-attention test and then fails the moment the sequence lengths
differ — so the score matrix is `(S_q, S_k)`, not square.

In [ ]:
# Colab setup (no-op when running locally).
# jax-judge is not published on PyPI, so the judge is installed from the
# repo itself. Regenerate with JAXCODE_REPO=you/YourFork to point this at
# your own fork:  JAXCODE_REPO=you/JAXCode make notebooks
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q flax optax')
    get_ipython().run_line_magic(
        'pip', 'install -q git+https://github.com/YOUR-GITHUB-USERNAME/JAXCode.git')
except ImportError:
    pass

In [ ]:
import jax
import jax.numpy as jnp
from flax import nnx

print("JAX", jax.__version__, "|", jax.devices())

In [ ]:
# ✏️ YOUR IMPLEMENTATION HERE

import jax
import jax.numpy as jnp
from flax import nnx


class MultiHeadAttention(nnx.Module):
    """Multi-head attention over (B, S, d_model)."""

    def __init__(self, d_model: int, num_heads: int, *, rngs: nnx.Rngs):
        pass  # Replace this

    def __call__(self, Q, K, V):
        """Q: (B, seq_q, d_model), K/V: (B, seq_k, d_model) -> (B, seq_q, d_model)"""
        pass  # Replace this

In [ ]:
# 🔍 Scratch cell — poke at your implementation
import jax
import jax.numpy as jnp
from flax import nnx

mha = MultiHeadAttention(32, 4, rngs=nnx.Rngs(params=0))

x = jax.random.normal(jax.random.key(1), (2, 6, 32))
print("self-attention :", mha(x, x, x).shape)

ctx = jax.random.normal(jax.random.key(2), (2, 10, 32))
print("cross-attention:", mha(x, ctx, ctx).shape, "(query length wins)")

In [ ]:
# ✅ SUBMIT — run this cell to check your solution
from jax_judge import check, hint, solution

check("mha")

# hint("mha")      # stuck? nudge without the answer
# solution("mha")  # spoiler: the reference implementation